In [10]:
import logging

from typing import Any
from uuid import uuid4

import httpx

from a2a.client import A2ACardResolver, A2AClient
from a2a.types import (
    AgentCard,
    MessageSendParams,
    SendMessageRequest,
    SendStreamingMessageRequest,
)
from a2a.utils.constants import (
    AGENT_CARD_WELL_KNOWN_PATH,
    EXTENDED_AGENT_CARD_PATH,
)


async def main() -> None:
    # Configure logging to show INFO level messages
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)  # Get a logger instance

    # --8<-- [start:A2ACardResolver]

    base_url = 'http://localhost:7000'

    async with httpx.AsyncClient() as httpx_client:
        # Initialize A2ACardResolver
        resolver = A2ACardResolver(
            httpx_client=httpx_client,
            base_url=base_url,
            # agent_card_path uses default, extended_agent_card_path also uses default
        )
        # --8<-- [end:A2ACardResolver]

        # Fetch Public Agent Card and Initialize Client
        final_agent_card_to_use: AgentCard | None = None

        try:
            logger.info(
                f'Attempting to fetch public agent card from: {base_url}{AGENT_CARD_WELL_KNOWN_PATH}'
            )
            _public_card = (
                await resolver.get_agent_card()
            )  # Fetches from default public path
            logger.info('Successfully fetched public agent card:')
            logger.info(
                _public_card.model_dump_json(indent=2, exclude_none=True)
            )
            final_agent_card_to_use = _public_card
            logger.info(
                '\nUsing PUBLIC agent card for client initialization (default).'
            )
        except Exception as e:
            logger.error(
                f'Critical error fetching public agent card: {e}', exc_info=True
            )
            raise RuntimeError(
                'Failed to fetch the public agent card. Cannot continue.'
            ) from e

        # --8<-- [start:send_message]
        client = A2AClient(
            httpx_client=httpx_client, agent_card=final_agent_card_to_use
        )
        logger.info('A2AClient initialized.')

        send_message_payload: dict[str, Any] = {
            'message': {
                'role': 'user',
                'parts': [
                    {'kind': 'text', 'text': 'what is weather today'}
                ],
                'message_id': uuid4().hex,
            },
        }
        request = SendMessageRequest(
            id=str(uuid4()), params=MessageSendParams(**send_message_payload)
        )

        response = await client.send_message(request)
        response.result
        print(response.model_dump(mode='json', exclude_none=True))
        print(response.model_dump()['result']['artifacts'][0]['parts'])





await main()

INFO:__main__:Attempting to fetch public agent card from: http://localhost:7000/.well-known/agent-card.json
INFO:httpx:HTTP Request: GET http://localhost:7000/.well-known/agent-card.json "HTTP/1.1 200 OK"
INFO:a2a.client.card_resolver:Successfully fetched agent card data from http://localhost:7000/.well-known/agent-card.json: {'capabilities': {'streaming': True}, 'defaultInputModes': ['text'], 'defaultOutputModes': ['text'], 'description': 'An agent that provides weather information based on user queries.', 'name': 'Weather Agent', 'preferredTransport': 'JSONRPC', 'protocolVersion': '0.3.0', 'skills': [{'description': 'Provides weather information for a given location.', 'id': 'weather_info', 'name': 'Weather Info', 'tags': ['weather']}], 'url': 'http://localhost:7000', 'version': '1.0.0'}
INFO:__main__:Successfully fetched public agent card:
INFO:__main__:{
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],

AttributeError: 'SendMessageResponse' object has no attribute 'result'

In [43]:
import logging

from typing import Any
from uuid import uuid4

import httpx

from a2a.client import A2ACardResolver, A2AClient
from a2a.types import (
    AgentCard,
    MessageSendParams,
    SendMessageRequest,
    SendStreamingMessageRequest,
)
from langchain_core.tools import StructuredTool
from a2a.utils.constants import (
    AGENT_CARD_WELL_KNOWN_PATH,
    EXTENDED_AGENT_CARD_PATH,
)

In [44]:
async def create_a2a_tools(base_url: str):
    async with httpx.AsyncClient() as httpx_client:

        # 1️⃣ Resolve agent card
        resolver = A2ACardResolver(
            httpx_client=httpx_client,
            base_url=base_url,
        )

        agent_card: AgentCard = await resolver.get_agent_card()

        # 2️⃣ Create A2A client
        client = A2AClient(
            httpx_client=httpx_client,
            agent_card=agent_card,
        )

        # 3️⃣ Create tools from skills
        tools = []
        for skill in agent_card.skills:
            tool = _create_tool_for_skill(client, skill)
            tools.append(tool)

        return tools
    
def _extract_text_from_response(response) -> str:
    parts = response.result.message.parts

    texts = []
    for part in parts:
        if getattr(part, "text", None):
            texts.append(part.text)

    return "\n".join(texts)

def _create_tool_for_skill(client: A2AClient, skill):

    async def _tool_func(query: str) -> str:
        request = SendMessageRequest(
            id=str(uuid4()),
            params=MessageSendParams(
                message={
                    "role": "user",
                    "parts": [
                        {
                            "kind": "text",
                            "text": query,
                        }
                    ],
                    "message_id": uuid4().hex,
                }
            ),
        )

        response = await client.send_message(request)
        return _extract_text_from_response(response)

    return StructuredTool.from_function(
        coroutine=_tool_func,
        name=skill.name,
        description=skill.description
        or f"A2A skill: {skill.name}",
    )


In [45]:
await create_a2a_tools("http://localhost:7000")

INFO:httpx:HTTP Request: GET http://localhost:7000/.well-known/agent-card.json "HTTP/1.1 200 OK"
INFO:a2a.client.card_resolver:Successfully fetched agent card data from http://localhost:7000/.well-known/agent-card.json: {'capabilities': {'streaming': True}, 'defaultInputModes': ['text'], 'defaultOutputModes': ['text'], 'description': 'An agent that provides weather information based on user queries.', 'name': 'Weather Agent', 'preferredTransport': 'JSONRPC', 'protocolVersion': '0.3.0', 'skills': [{'description': 'Provides weather information for a given location.', 'id': 'weather_info', 'name': 'Weather Info', 'tags': ['weather']}], 'url': 'http://localhost:7000', 'version': '1.0.0'}
/tmp/ipykernel_1064/2785060079.py:13: DeprecationWarning: A2AClient is deprecated and will be removed in a future version. Use ClientFactory to create a client with a JSON-RPC transport.
  client = A2AClient(


[StructuredTool(name='Weather Info', description='Provides weather information for a given location.', args_schema=<class 'langchain_core.utils.pydantic.Weather Info'>, coroutine=<function _create_tool_for_skill.<locals>._tool_func at 0x777cc634ee80>)]